# Create the wave metamodel

Pipeline (all settings come from ``NorthCarolina.json`` → ``metamodel``).
Station QC runs **automatically** inside ``utils.metamodel`` — no extra notebook steps.

1. **Train / load** models (`fit_models`) — scans SnapWave static cases, excludes bad sites, then MDA + PCA + GP per goal.
2. **Bulk reconstruction** (`reconstruct_bulk`) — hs/tp/wd/ds; skips QC-failed stations.
3. **Zero-fill** missing forcing (`apply_missing_forcing_zeros`).
4. **Spectral reconstruction** (`reconstruct_spectral`) — sums partitions and QC-filters hs/tp/dm.
5. **Save** (`save_spectral`) → `outputs/spectral_metamodel.nc`.

In [1]:
from utils import metamodel as mm
from utils.config import GOALS, METAMODEL, reload

reload()  # pick up edits to NorthCarolina.json without restarting kernel
print(f"goals            : {len(GOALS)}")
print(f"MDA points       : {METAMODEL['num_mda_points']}")
print(f"spectral output  : {mm.spectral_output_path()}")

goals            : 14
MDA points       : 450
spectral output  : /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc


## Parameters

Tweak these while experimenting — especially `time` and `stations` for the
spectral step.

In [2]:
# --- training --------------------------------------------------------------
force_retrain = False  # set True to overwrite existing .pkl models

# --- bulk reconstruction ---------------------------------------------------
time_bulk = "1986-01-15"  # any xarray time selector (month, day, slice, ...)

# --- spectral reconstruction (bottleneck) ------------------------------------
time_spec = None  # None → use full bulk window; or e.g. "1986-01-15"
sites = "isobath"  # flexible: int, slice, list, "buoy", "isobath", name
# sites = slice(0, 3)   # quick smoke test on first 3 sites
# sites = -1              # single site by index

## 1 · Train / load models

In [3]:
models = mm.fit_models(force_retrain=force_retrain)
len(models["mda"]), len(models["pca"]), len(models["gp"])

2026-06-17 14:46:16,619 - NorthCarolina.metamodel - INFO - Fitting models for 14 goal(s)  (num_mda_points=450, force_retrain=False)
2026-06-17 14:46:16,826 - NorthCarolina.metamodel - INFO - Goal  1: loaded existing models
2026-06-17 14:46:16,863 - NorthCarolina.metamodel - INFO - Goal  2: loaded existing models
2026-06-17 14:46:16,901 - NorthCarolina.metamodel - INFO - Goal  3: loaded existing models
2026-06-17 14:46:16,935 - NorthCarolina.metamodel - INFO - Goal  4: loaded existing models
2026-06-17 14:46:16,967 - NorthCarolina.metamodel - INFO - Goal  5: loaded existing models
2026-06-17 14:46:17,002 - NorthCarolina.metamodel - INFO - Goal  6: loaded existing models
2026-06-17 14:46:17,034 - NorthCarolina.metamodel - INFO - Goal  7: loaded existing models
2026-06-17 14:46:17,068 - NorthCarolina.metamodel - INFO - Goal  8: loaded existing models
2026-06-17 14:46:17,208 - NorthCarolina.metamodel - INFO - Goal  9: loaded existing models
2026-06-17 14:46:17,240 - NorthCarolina.metamodel

(14, 14, 14)

## 2 · Bulk reconstruction

In [4]:
bulk = mm.reconstruct_bulk(time_bulk, models=models)
print(f"size : {bulk.nbytes * 1e-9:.2f} GB")
bulk

2026-06-17 14:46:17,880 - NorthCarolina.metamodel - INFO - Bulk reconstruction: time='1986-01-15', goals=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], partitions=['bulk', 'sea', 'swell1', 'swell2', 'swell3']
2026-06-17 14:46:17,886 - NorthCarolina.metamodel - INFO - Goal  1 / bulk   : reconstructing 24 / 24 timesteps
2026-06-17 14:46:18,333 - NorthCarolina.metamodel - INFO - Goal  2 / bulk   : reconstructing 24 / 24 timesteps
INFO:ExactGPInterpolation:Preprocessing dataset for prediction
INFO:ExactGPInterpolation:Checking for NaNs in subset data
INFO:ExactGPInterpolation:Preprocessing subset data
INFO:ExactGPInterpolation:Normalizing subset data
INFO:ExactGPInterpolation:Proposed max custom scaler for ds is lower than datapoint
INFO:ExactGPInterpolation:Proposed max custom scaler for wd_v is lower than datapoint
INFO:ExactGPInterpolation:Subset data preprocessed successfully
INFO:ExactGPInterpolation:Predicting target variable: PC1
INFO:ExactGPInterpolation:Predicting target variabl

size : 0.01 GB


<xarray.Dataset> Size: 12MB
Dimensions:      (time: 24, goal: 14, partition: 5, sites: 216)
Coordinates:
  * time         (time) datetime64[ns] 192B 1986-01-15 ... 1986-01-15T23:00:00
  * goal         (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude     (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude    (goal) float64 112B -75.5 -75.25 -75.06 ... -78.25 -78.62 -79.0
    seapoint     (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06 1.903e+06
  * partition    (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
  * sites        (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
Data variables:
    hs           (goal, partition, time, sites) float64 3MB 0.2157 ... nan
    tp           (goal, partition, time, sites) float64 3MB 6.956 6.956 ... nan
    wd           (goal, partition, time, sites) float64 3MB 20.15 22.73 ... nan
    ds           (goal, partition, time, sites) float64 3MB 31.4 31.4 ... nan

## 2b · Zero-fill missing forcing

Sea/swell partitions can be NaN when a component is absent. The metamodel only
predicts valid timesteps; this step sets those gaps to ``0.0`` before the
spectral sum.

In [5]:
bulk = mm.apply_missing_forcing_zeros(bulk)
bulk

2026-06-17 14:46:24,542 - NorthCarolina.metamodel - INFO - Zero-filled 230688 NaN values in hs, tp, wd, ds


<xarray.Dataset> Size: 12MB
Dimensions:      (time: 24, goal: 14, partition: 5, sites: 216)
Coordinates:
  * time         (time) datetime64[ns] 192B 1986-01-15 ... 1986-01-15T23:00:00
  * goal         (goal) int64 112B 1 2 3 4 5 6 7 8 9 10 11 12 13 14
    latitude     (goal) float64 112B 36.56 36.12 35.69 ... 33.19 33.0 32.88
    longitude    (goal) float64 112B -75.5 -75.25 -75.06 ... -78.25 -78.62 -79.0
    seapoint     (goal) float64 112B 1.904e+06 1.904e+06 ... 1.903e+06 1.903e+06
  * partition    (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
  * sites        (sites) <U12 10kB '5m_10' '5m_11' ... 'buoy_8658120'
    site_source  (sites) <U7 6kB 'isobath' 'isobath' 'isobath' ... 'buoy' 'buoy'
    site_label   (sites) <U10 9kB '5m' '5m' '5m' ... 'levels' 'levels' 'levels'
    site_depth   (sites) float64 2kB -5.0 -5.0 -5.0 -5.0 ... nan nan nan nan
Data variables:
    hs           (goal, partition, time, sites) float64 3MB 0.2157 ... 0.0
    tp           (goal, partition, time, sites) float64 3MB 6.956 6.956 ... 0.0
    wd           (goal, partition, time, sites) float64 3MB 20.15 22.73 ... 0.0
    ds           (goal, partition, time, sites) float64 3MB 31.4 31.4 ... 0.0

## 3 · Spectral reconstruction

Start with a **small** `sites` selection above, then scale up.

In [6]:
spectral = mm.reconstruct_spectral(bulk, sites=sites, time=time_spec)
spectral

2026-06-17 14:46:24,587 - NorthCarolina.metamodel - INFO - Spectral reconstruction: 185 site(s), partitions=['sea', 'swell1', 'swell2', 'swell3'], time=None
2026-06-17 14:46:24,587 - NorthCarolina.metamodel - INFO - Spectral site   1 / 185: 5m_10
2026-06-17 14:46:24,850 - NorthCarolina.metamodel - INFO - Spectral site   2 / 185: 5m_11
2026-06-17 14:46:25,034 - NorthCarolina.metamodel - INFO - Spectral site   3 / 185: 5m_12
2026-06-17 14:46:25,187 - NorthCarolina.metamodel - INFO - Spectral site   4 / 185: 5m_13
2026-06-17 14:46:25,356 - NorthCarolina.metamodel - INFO - Spectral site   5 / 185: 5m_14
2026-06-17 14:46:25,518 - NorthCarolina.metamodel - INFO - Spectral site   6 / 185: 5m_15
2026-06-17 14:46:25,673 - NorthCarolina.metamodel - INFO - Spectral site   7 / 185: 5m_16
2026-06-17 14:46:25,849 - NorthCarolina.metamodel - INFO - Spectral site   8 / 185: 5m_17
2026-06-17 14:46:26,034 - NorthCarolina.metamodel - INFO - Spectral site   9 / 185: 5m_18
2026-06-17 14:46:26,202 - NorthCa

<xarray.Dataset> Size: 102kB
Dimensions:      (sites: 182, time: 24)
Coordinates:
  * sites        (sites) object 1kB '5m_10' '5m_11' ... '15m_82' '15m_83'
    site         (sites) <U6 4kB '5m_10' '5m_11' '5m_12' ... '15m_82' '15m_83'
    site_label   (sites) <U10 7kB '5m' '5m' '5m' '5m' ... '15m' '15m' '15m'
    site_depth   (sites) float64 1kB -5.0 -5.0 -5.0 -5.0 ... -15.0 -15.0 -15.0
  * time         (time) datetime64[ns] 192B 1986-01-15 ... 1986-01-15T23:00:00
    site_source  <U7 28B 'isobath'
Data variables:
    hs           (sites, time) float64 35kB 0.3299 0.3365 0.3771 ... 1.211 1.158
    tp           (sites, time) float32 17kB 7.149 7.253 6.798 ... 6.297 6.387
    dm           (sites, time) float64 35kB 31.76 35.73 14.45 ... 79.88 81.14
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

## 4 · Save

In [7]:
out_path = mm.save_spectral(spectral)
print(f"wrote {out_path}")
print(f"excluded stations: {mm.get_excluded_sites()}")

2026-06-17 14:46:57,333 - NorthCarolina.metamodel - INFO - Wrote spectral metamodel to /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc (182 site(s), 0.10 MB)


wrote /home/tausiaj/GitHub-GeoOcean/HyGloW/NorthCarolina/outputs/spectral_metamodel.nc
excluded stations: ['10m_25', '10m_43', '5m_45']
